# Explore raw Steam data

Quick look at `steam_reviews.json.gz` / `steam_games.json.gz` (Kang & McAuley's SASRec Steam dataset, http://cseweb.ucsd.edu/~wckang/) before writing any cleaning logic.

**What's confirmed vs. inferred here:**
- The **games** file schema below is copied verbatim from the official SASRec repo README example — reliable.
- The **reviews** file schema is *not* documented anywhere I could verify, so this notebook doesn't assume it. It parses the raw records, prints whatever keys are actually there, and auto-detects plausible user/item/time columns from that — the peek and stats below are the source of truth, not the guesses in this markdown cell.
- Both files are **"loose" JSON** (Python dict literals, single-quoted) rather than strict JSON — the same format as the pre-2023 Amazon dumps. The dataset's own reference parser uses `eval()` per line; this notebook uses `ast.literal_eval` instead, which reads the same loose-dict syntax without executing arbitrary code.

# 0. Import

In [1]:
import ast
import gzip
import itertools
import pandas as pd
from local_package.config.data import STEAM_RAW_DIR

# 1. Configuration

In [2]:
# Path
DATA_DIR = STEAM_RAW_DIR

REVIEWS_FILE = DATA_DIR / "steam_reviews.json.gz"
GAMES_FILE = DATA_DIR / "steam_games.json.gz"

In [3]:
# Records peeking
N_PEEK = 3  # how many raw records to pretty-print
N_SAMPLE = 1000  # rows to load into a DataFrame; None = load everything

In [4]:
def iter_loose_json_gz(path):
    """Yield one parsed record per line. These dumps are 'loose' JSON (Python dict literals,
    single-quoted) rather than strict JSON, so plain json.loads fails on them."""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            yield ast.literal_eval(line)

In [5]:
def load_loose_json_gz(path, n_rows=None):
    records = itertools.islice(iter_loose_json_gz(path), n_rows) if n_rows else iter_loose_json_gz(path)
    return pd.DataFrame.from_records(records)

## Raw record peek

In [6]:
print(f"--- first {N_PEEK} raw review records ---")
for rec in itertools.islice(iter_loose_json_gz(REVIEWS_FILE), N_PEEK):
    print(rec)
    print()

--- first 3 raw review records ---
{'username': 'Chaos Syren', 'hours': 0.1, 'products': 41, 'product_id': '725280', 'page_order': 0, 'date': '2017-12-17', 'text': 'This would not be acceptable as an entertainment even back in the day when these graphics were all there was to be had. No effort has been made to bring the player into any story or even entertain.', 'early_access': False, 'page': 1}

{'username': '₮ʜᴇ Wᴀʀᴛᴏɴ', 'hours': 51.1, 'products': 769, 'product_id': '328100', 'page_order': 0, 'date': '2017-12-27', 'text': 'looks like a facebook game', 'early_access': False, 'page': 1}

{'username': 'hello?<', 'text': 'Better than Minecraft', 'hours': 14.6, 'date': '2017-10-16', 'early_access': False, 'found_funny': 2, 'product_id': '328100', 'page_order': 1, 'compensation': 'Product received for free', 'products': 2, 'page': 1}



In [7]:
print(f"--- first {N_PEEK} raw game records ---")
for rec in itertools.islice(iter_loose_json_gz(GAMES_FILE), N_PEEK):
    print(rec)
    print()

--- first 3 raw game records ---
{'publisher': 'Kotoshiro', 'genres': ['Action', 'Casual', 'Indie', 'Simulation', 'Strategy'], 'app_name': 'Lost Summoner Kitty', 'title': 'Lost Summoner Kitty', 'url': 'http://store.steampowered.com/app/761140/Lost_Summoner_Kitty/', 'release_date': '2018-01-04', 'tags': ['Strategy', 'Action', 'Indie', 'Casual', 'Simulation'], 'discount_price': 4.49, 'reviews_url': 'http://steamcommunity.com/app/761140/reviews/?browsefilter=mostrecent&p=1', 'specs': ['Single-player'], 'price': 4.99, 'early_access': False, 'id': '761140', 'developer': 'Kotoshiro'}

{'publisher': 'Making Fun, Inc.', 'genres': ['Free to Play', 'Indie', 'RPG', 'Strategy'], 'app_name': 'Ironbound', 'sentiment': 'Mostly Positive', 'title': 'Ironbound', 'url': 'http://store.steampowered.com/app/643980/Ironbound/', 'release_date': '2018-01-04', 'tags': ['Free to Play', 'Strategy', 'Indie', 'RPG', 'Card Game', 'Trading Card Game', 'Turn-Based', 'Fantasy', 'Tactical', 'Dark Fantasy', 'Board Game',

## Load into DataFrames

Check `review_df.columns` against the field names assumed anywhere else in the pipeline (`preprocess_steam.ipynb`, `item_cf_steam.ipynb`) before trusting them — they were not verified against a primary source.

In [8]:
review_df = load_loose_json_gz(REVIEWS_FILE, N_SAMPLE)
game_df = load_loose_json_gz(GAMES_FILE, N_SAMPLE)
print("reviews:", review_df.shape)
print("games:", game_df.shape)
print("review columns:", list(review_df.columns))
print("game columns:", list(game_df.columns))
review_df.head(3)

reviews: (1000, 12)
games: (1000, 16)
review columns: ['username', 'hours', 'products', 'product_id', 'page_order', 'date', 'text', 'early_access', 'page', 'found_funny', 'compensation', 'user_id']
game columns: ['publisher', 'genres', 'app_name', 'title', 'url', 'release_date', 'tags', 'discount_price', 'reviews_url', 'specs', 'price', 'early_access', 'id', 'developer', 'sentiment', 'metascore']


,username,hours,products,product_id,page_order,date,text,early_access,page,found_funny,compensation,user_id
0,Chaos Syren,0.1,41,725280,0,2017-12-17,This would not be acceptable as an entertainme...,False,1,NaN,NaN,NaN
1,₮ʜᴇ Wᴀʀᴛᴏɴ,51.1,769,328100,0,2017-12-27,looks like a facebook game,False,1,NaN,NaN,NaN
2,hello?<,14.6,2,328100,1,2017-10-16,Better than Minecraft,False,1,2.0,Product received for free,NaN


In [9]:
game_df.head(3)

,publisher,genres,app_name,title,url,release_date,tags,discount_price,reviews_url,specs,price,early_access,id,developer,sentiment,metascore
0,Kotoshiro,"[Action, Casual, Indie, Simulation, Strategy]",Lost Summoner Kitty,Lost Summoner Kitty,http://store.steampowered.com/app/761140/Lost_...,2018-01-04,"[Strategy, Action, Indie, Casual, Simulation]",4.49,http://steamcommunity.com/app/761140/reviews/?...,[Single-player],4.99,False,761140,Kotoshiro,NaN,NaN
1,"Making Fun, Inc.","[Free to Play, Indie, RPG, Strategy]",Ironbound,Ironbound,http://store.steampowered.com/app/643980/Ironb...,2018-01-04,"[Free to Play, Strategy, Indie, RPG, Card Game...",NaN,http://steamcommunity.com/app/643980/reviews/?...,"[Single-player, Multi-player, Online Multi-Pla...",Free To Play,False,643980,Secret Level SRL,Mostly Positive,NaN
2,Poolians.com,"[Casual, Free to Play, Indie, Simulation, Sports]",Real Pool 3D - Poolians,Real Pool 3D - Poolians,http://store.steampowered.com/app/670290/Real_...,2017-07-24,"[Free to Play, Simulation, Sports, Casual, Ind...",NaN,http://steamcommunity.com/app/670290/reviews/?...,"[Single-player, Multi-player, Online Multi-Pla...",Free to Play,False,670290,Poolians.com,Mostly Positive,NaN


## Review stats

In [10]:
print("missing values per column:")
print(review_df.isna().mean().sort_values(ascending=False))

missing values per column:
compensation    0.939
found_funny     0.793
user_id         0.602
hours           0.001
product_id      0.000
products        0.000
username        0.000
page_order      0.000
early_access    0.000
text            0.000
date            0.000
page            0.000
dtype: float64


In [11]:
# common candidate names for the user/item/time columns in this dataset family — confirmed
# against the printed column list above, not assumed
USER_COL_CANDIDATES = ["user_id", "username", "userid"]
ITEM_COL_CANDIDATES = ["product_id", "item_id", "id"]
TIME_COL_CANDIDATES = ["date", "timestamp", "posted"]

def first_present(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

user_col = first_present(review_df, USER_COL_CANDIDATES)
item_col = first_present(review_df, ITEM_COL_CANDIDATES)
time_col = first_present(review_df, TIME_COL_CANDIDATES)
print("detected columns -> user:", user_col, "| item:", item_col, "| time:", time_col)

detected columns -> user: user_id | item: product_id | time: date


In [12]:
if time_col:
    print(f"'{time_col}' sample values:", review_df[time_col].dropna().head(5).tolist())
else:
    print("no time-like column detected; inspect review_df.columns manually")

'date' sample values: ['2017-12-17', '2017-12-27', '2017-10-16', '2018-01-04', '2018-01-04']


In [13]:
if user_col and item_col:
    dup_pairs = review_df.duplicated(subset=[user_col, item_col]).sum()
    print(f"duplicated ({user_col}, {item_col}) rows: {dup_pairs} / {len(review_df)}")
else:
    print("could not detect both a user and an item column; inspect review_df.columns manually")

duplicated (user_id, product_id) rows: 566 / 1000


In [14]:
if user_col and item_col:
    inter_per_user = review_df.groupby(user_col).size()
    inter_per_item = review_df.groupby(item_col).size()
    print("interactions per user:\n", inter_per_user.describe())
    print("\ninteractions per item:\n", inter_per_item.describe())

interactions per user:
 count    391.000000
mean       1.017903
std        0.195295
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        4.000000
dtype: float64

interactions per item:
 count    37.000000
mean     27.027027
std      19.653225
min       1.000000
25%       9.000000
50%      23.000000
75%      50.000000
max      59.000000
dtype: float64


## Games stats

In [15]:
print("missing values per column:")
print(game_df.isna().mean().sort_values(ascending=False))

missing values per column:
discount_price    0.984
metascore         0.594
sentiment         0.096
price             0.044
genres            0.040
publisher         0.026
developer         0.024
tags              0.015
release_date      0.012
title             0.011
specs             0.004
app_name          0.001
id                0.001
reviews_url       0.001
url               0.000
early_access      0.000
dtype: float64


In [16]:
print("price stats (non-null):")
print(game_df["price"].describe())
print(f"missing price: {game_df['price'].isna().mean():.1%}")

price stats (non-null):
count     956.00
unique     35.00
top         9.99
freq      243.00
Name: price, dtype: float64
missing price: 4.4%


In [17]:
flat_genres = game_df["genres"].explode()
print("top genres:")
print(flat_genres.value_counts().head(20))

top genres:
genres
Action                      381
Strategy                    255
Indie                       252
Casual                      165
Adventure                   162
RPG                         154
Simulation                  152
Racing                       33
Free to Play                 25
Massively Multiplayer        21
Sports                       15
Early Access                  6
Animation &amp; Modeling      1
Video Production              1
Name: count, dtype: int64


## Reviews ↔ games join coverage

In [18]:
if item_col:
    coverage = review_df[item_col].astype(str).isin(game_df["id"].astype(str)).mean()
    print(f"reviews whose {item_col} has game metadata: {coverage:.1%}")
else:
    print("no item column detected; can't check join coverage")

reviews whose product_id has game metadata: 10.8%
